# Dashboard Ruang Sales — Revisi Final

Notebook ini memuat **kode lengkap `scripts/dashboard_model.py` yang sudah direvisi**, berasal dari teks file yang kamu kirim.

**Perubahan:**
- Seluruh kotak **Insight utama** di halaman Ringkasan sudah dihapus.
- Heatmap ARI di halaman Ringkasan dan Rekomendasi serta kartu ARI di Metodologi sudah dihapus.
- Bagian “Hasil klasterisasi” di Ringkasan dibuat lebar penuh.
- Fungsi dashboard lainnya dan hasil perhitungan model tidak diubah.

**Cara penggunaan di Google Colab:** jalankan semua sel, lalu unduh `dashboard_model.py` dari output. Ganti file di folder **`scripts/`** pada GitHub, dan commit. **Jangan mengganti `app.py`**.

> ARI tidak memerlukan ground truth bila dipakai untuk membandingkan dua partisi hasil klasterisasi, tetapi bukan ukuran akurasi terhadap label sebenarnya. Komponen ARI dihapus dari tampilan atas permintaanmu.

## 1 — Kode dashboard hasil revisi

Kode di bawah sudah disimpan di dalam notebook. Tidak perlu mengunggah `dashboard_model.py` lagi.

In [1]:
# Sumber kode lengkap; tiap baris ditulis sebagai string untuk dibangun kembali tanpa mengimpor Streamlit di Colab.
dashboard_code = (
    '"""Dashboard Streamlit Ruang Sales — membaca hasil model dari notebook Colab (hasil_model/).\n'
    '\n'
    'Tidak ada pelatihan ulang di dashboard: semua label, centroid, metrik, dan koordinat peta sudah\n'
    'dihitung di Colab (FastDTW · 3 representasi · 6 algoritma). Alurnya tetap seperti R Shiny:\n'
    'pengguna memilih representasi, algoritma, dan parameter, lalu menekan **Tampilkan hasil**.\n'
    '"""\n'
    'from __future__ import annotations\n'
    '\n'
    'import io\n'
    'import zipfile\n'
    'from itertools import combinations\n'
    'from typing import Any\n'
    '\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'import streamlit as st\n'
    '\n'
    'import config\n'
    'from scripts import interactive as ix\n'
    'from scripts import interactive_viz as iv\n'
    'from scripts import model_store as ms\n'
    'from scripts import visualization as V\n'
    '\n'
    'PAGES = ["Ringkasan", "Eksplorasi Data", "Eksplorasi Jarak", "Klasterisasi", "Rekomendasi Metode", "Metodologi"]\n'
    'NAV_SVG = ["grid", "table", "trend", "network", "trophy", "info"]\n'
    'ICONS = ["space_dashboard", "query_stats", "timeline", "bubble_chart", "workspace_premium", "menu_book"]\n'
    'REP_LABEL = ix.REPRESENTATIONS\n'
    'ALGO_LABEL = ix.ALGORITHMS\n'
    'Y_TITLE = {"original": "Penjualan asli", "minmax": "Nilai Min-Max", "zscore": "Nilai z-score"}\n'
    '\n'
    '\n'
    '# =======================================================================================\n'
    '# Data\n'
    '# =======================================================================================\n'
    '@st.cache_resource(show_spinner="Memuat hasil model…")\n'
    'def load_bundle(mtime: float) -> dict:\n'
    '    """Memuat model_results.json + data turunan (di-cache; ``mtime`` memaksa muat ulang bila file berubah)."""\n'
    '    res = ms.load_results(config.RESULTS_FILE)\n'
    '    W = np.asarray(res["W"], dtype=float)\n'
    '    codes = np.asarray(res["codes"]).astype(str)\n'
    '    sales = pd.DataFrame(W, columns=config.WEEK_COLUMNS[: W.shape[1]]).assign(Product_Code=codes)\n'
    '    from scripts.data import build_product_statistics, build_weekly_statistics\n'
    '    return {\n'
    '        "res": res, "W": W, "codes": codes,\n'
    '        "X": {rep: ix.represent(W, rep) for rep in res["spaces"]},\n'
    '        "stats": build_product_statistics(sales, "Product_Code", config.WEEK_COLUMNS[: W.shape[1]]),\n'
    '        "weekly": build_weekly_statistics(sales, config.WEEK_COLUMNS[: W.shape[1]]),\n'
    '        "table": ms.metrics_table(res),\n'
    '    }\n'
    '\n'
    '\n'
    'def bundle() -> dict | None:\n'
    '    if not config.RESULTS_FILE.is_file():\n'
    '        return None\n'
    '    return load_bundle(config.RESULTS_FILE.stat().st_mtime)\n'
    '\n'
    '\n'
    'def result(b: dict, key: str):\n'
    '    rep, algo, _, extra = key.split("|")\n'
    '    return ms.run_to_result(b["res"]["runs"][key], b["res"]["spaces"][rep], linkage=extra if algo == "agglomerative" else None)\n'
    '\n'
    '\n'
    'def key_label(key: str) -> str:\n'
    '    rep, algo, k, extra = key.split("|")\n'
    '    tail = f"eps p{extra}" if algo == "dbscan" else f"K = {k}" + (f" · {extra}" if extra else "")\n'
    '    return f"{ALGO_LABEL[algo].split(\' (\')[0]} · {REP_LABEL[rep].split(\' (\')[0]} · {tail}"\n'
    '\n'
    '\n'
    'def active_key(b: dict) -> str:\n'
    '    key = st.session_state.get("active_key")\n'
    '    if key not in b["res"]["runs"]:\n'
    '        best = b["res"]["best"]\n'
    '        key = best.get(config.DEFAULT_REPRESENTATION) or next(v for v in best.values() if v) or next(iter(b["res"]["runs"]))\n'
    '        st.session_state["active_key"] = key\n'
    '    return key\n'
    '\n'
    '\n'
    '# =======================================================================================\n'
    '# Komponen tampilan\n'
    '# =======================================================================================\n'
    'def apply_css() -> None:\n'
    '    css = config.PROJECT_ROOT / "assets" / "style.css"\n'
    '    if css.exists():\n'
    '        st.markdown(f"<style>{css.read_text(encoding=\'utf-8\')}</style>", unsafe_allow_html=True)\n'
    '\n'
    '\n'
    'def _glow(fig) -> None:\n'
    '    """Efek neon: garis utama diberi bayangan lebar-transparan. Murni tampilan; data & hover tidak berubah."""\n'
    '    try:\n'
    '        import plotly.graph_objects as go\n'
    '        traces = list(fig.data)\n'
    '        if len(traces) > 14:\n'
    '            return\n'
    '        extra = []\n'
    '        for t in traces:\n'
    '            if getattr(t, "type", "") != "scatter" or "lines" not in (t.mode or "") or getattr(t, "stackgroup", None):\n'
    '                continue\n'
    '            color, width, dash = t.line.color, t.line.width or 2, t.line.dash\n'
    '            if not isinstance(color, str) or width < 2 or dash not in (None, "solid") or t.x is None or len(t.x) > 400:\n'
    '                continue\n'
    '            extra.append(go.Scatter(x=t.x, y=t.y, mode="lines", line=dict(color=color, width=width * 4, shape=t.line.shape), opacity=0.16,\n'
    '                                    hoverinfo="skip", showlegend=False, xaxis=t.xaxis, yaxis=t.yaxis))\n'
    '        if extra:\n'
    '            fig.add_traces(extra)\n'
    '    except Exception:\n'
    '        pass\n'
    '\n'
    '\n'
    'def plot(fig, key: str) -> None:\n'
    '    fig.update_layout(title=None)\n'
    '    _glow(fig)\n'
    '    st.plotly_chart(fig, width="stretch", theme=None, key=key,\n'
    '                    config={"displaylogo": False, "displayModeBar": "hover", "modeBarButtonsToRemove": ["select2d", "lasso2d"]})\n'
    '\n'
    '\n'
    '_SVG = {\n'
    '    "box": \'<path d="M21 8l-9-5-9 5 9 5 9-5z"/><path d="M3 8v8l9 5 9-5V8"/><path d="M12 13v8"/>\',\n'
    '    "layers": \'<path d="M12 3l9 5-9 5-9-5 9-5z"/><path d="M3 13l9 5 9-5"/>\',\n'
    '    "bars": \'<path d="M4 20V10"/><path d="M10 20V4"/><path d="M16 20v-7"/><path d="M22 20H2"/>\',\n'
    '    "network": \'<circle cx="12" cy="5" r="2.2"/><circle cx="5" cy="18" r="2.2"/><circle cx="19" cy="18" r="2.2"/><circle cx="12" cy="12" r="2"/><path d="M12 7.2V10M10.4 13.2l-3.9 3.2M13.6 13.2l3.9 3.2"/>\',\n'
    '    "calendar": \'<rect x="3" y="5" width="18" height="16" rx="2"/><path d="M3 10h18M8 3v4M16 3v4"/>\',\n'
    '    "trend": \'<path d="M3 17l6-6 4 4 8-8"/><path d="M15 7h6v6"/>\',\n'
    '    "trend_down": \'<path d="M3 7l6 6 4-4 8 8"/><path d="M15 17h6v-6"/>\',\n'
    '    "cube": \'<path d="M12 2l9 5v10l-9 5-9-5V7z"/><path d="M12 22V12M21 7l-9 5-9-5"/>\',\n'
    '    "filter": \'<path d="M3 4h18l-7 9v6l-4 2v-8z"/>\',\n'
    '    "trophy": \'<path d="M8 21h8M12 17v4M7 4h10v5a5 5 0 01-10 0z"/><path d="M17 5h3a3 3 0 01-3 4M7 5H4a3 3 0 003 4"/>\',\n'
    '    "wave": \'<path d="M2 12c2.5-5 5-5 7.5 0s5 5 7.5 0 3.5-3 5-2"/>\',\n'
    '    "target": \'<circle cx="12" cy="12" r="8"/><circle cx="12" cy="12" r="4"/><circle cx="12" cy="12" r="1"/>\',\n'
    '    "scale": \'<path d="M12 3v18M5 21h14M5 7h14"/><path d="M5 7l-3 7a3 3 0 006 0zM19 7l-3 7a3 3 0 006 0z"/>\',\n'
    '    "grid": \'<rect x="3" y="3" width="7" height="7" rx="1.5"/><rect x="14" y="3" width="7" height="7" rx="1.5"/><rect x="3" y="14" width="7" height="7" rx="1.5"/><rect x="14" y="14" width="7" height="7" rx="1.5"/>\',\n'
    '    "zero": \'<circle cx="12" cy="12" r="8"/><path d="M6.5 17.5l11-11"/>\',\n'
    '    "sliders": \'<path d="M4 6h10M18 6h2M4 12h4M12 12h8M4 18h12M20 18h0"/><circle cx="16" cy="6" r="2"/><circle cx="10" cy="12" r="2"/><circle cx="18" cy="18" r="2"/>\',\n'
    '    "table": \'<rect x="3" y="4" width="18" height="16" rx="2"/><path d="M3 10h18M9 4v16"/>\',\n'
    '    "map": \'<circle cx="7" cy="8" r="2.5"/><circle cx="16" cy="7" r="2"/><circle cx="12" cy="16" r="3"/><path d="M3 21h18"/>\',\n'
    '    "pulse": \'<path d="M3 12h4l3-7 4 14 3-7h4"/>\',\n'
    '    "info": \'<circle cx="12" cy="12" r="9"/><path d="M12 8h0M11 12h1v5h1"/>\',\n'
    '}\n'
    '\n'
    '\n'
    'def icon(name: str, size: int = 20) -> str:\n'
    '    """Ikon garis sederhana (SVG inline, warna mengikuti currentColor)."""\n'
    '    return (f\'<svg width="{size}" height="{size}" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.8" \'\n'
    '            f\'stroke-linecap="round" stroke-linejoin="round">{_SVG.get(name, _SVG["info"])}</svg>\')\n'
    '\n'
    '\n'
    'def idn(value, digits: int = 0) -> str:\n'
    '    """Format angka Indonesia: titik ribuan, koma desimal (375.287 · 66,6)."""\n'
    '    if value is None or (isinstance(value, float) and not np.isfinite(value)):\n'
    '        return "–"\n'
    '    return f"{value:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")\n'
    '\n'
    '\n'
    'def kpis(cards: list[tuple]) -> None:\n'
    '    """cards = (label, nilai, keterangan[, ikon[, warna aksen]])."""\n'
    '    for col, card in zip(st.columns(len(cards)), cards):\n'
    '        label, value, detail = card[:3]\n'
    '        ic = card[3] if len(card) > 3 else "pulse"\n'
    '        accent = card[4] if len(card) > 4 else "var(--cyan)"\n'
    '        col.markdown(f\'<div class="kpi" style="--accent:{accent}"><div class="ic">{icon(ic)}</div><div><div class="kpi-label">{label}</div>\'\n'
    '                     f\'<div class="kpi-value">{value}</div><div class="kpi-detail">{detail}</div></div></div>\', unsafe_allow_html=True)\n'
    '\n'
    '\n'
    'def section(title: str, sub: str = "", ic: str = "pulse") -> None:\n'
    '    """Judul section konsisten: ikon cyan + judul + keterangan singkat."""\n'
    '    st.markdown(f\'<div class="section-title">{icon(ic)}<span>{title}</span></div>\' + (f\'<div class="section-sub">{sub}</div>\' if sub else ""),\n'
    '                unsafe_allow_html=True)\n'
    '\n'
    '\n'
    'def best_cards(b: dict, keys: dict) -> None:\n'
    '    """Tiga kartu model terbaik (Data asli · Min-Max · Scale) yang mudah dibandingkan."""\n'
    '    style = {"original": ("var(--emerald)", "bars"), "minmax": ("var(--gold)", "wave"), "zscore": ("var(--cyan)", "cube")}\n'
    '    cols = st.columns(len(keys))\n'
    '    for col, (rep, key) in zip(cols, keys.items()):\n'
    '        accent, ic = style.get(rep, ("var(--cyan)", "target"))\n'
    '        m = b["res"]["runs"][key]["metrics"]\n'
    '        col.markdown(\n'
    '            f\'<div class="best" style="--accent:{accent}"><div class="bg">{icon(ic, 44)}</div>\'\n'
    '            f\'<div class="t">{icon("trophy", 16)} Terbaik · {REP_LABEL[rep]}</div><div class="m">{key_label(key)}</div>\'\n'
    '            f\'<div class="s"><span>Silhouette <b>{idn(m["Silhouette"], 3)}</b></span><span>Balance <b>{idn(m["Balance"], 3)}</b></span>\'\n'
    '            f\'<span>η² nol <b>{idn(m["Eta2_nol"], 3)}</b></span></div></div>\', unsafe_allow_html=True)\n'
    '\n'
    '\n'
    '# =======================================================================================\n'
    '# Profil klaster untuk pembaca bisnis (murni turunan label + penjualan asli; tidak mengubah model)\n'
    '# =======================================================================================\n'
    'def cluster_story(b: dict, key: str) -> tuple[Any, pd.DataFrame]:\n'
    '    """Profil per cluster + julukan bisnis berdasarkan peringkat rata-rata penjualan mingguan."""\n'
    '    r = result(b, key)\n'
    '    prof = pd.DataFrame(ix.cluster_profile_rows(r, b["W"], b["codes"]))\n'
    '    real = prof[prof["Cluster"] != -1].sort_values("Rata2_Mingguan", ascending=False)\n'
    '    tag = {int(c): "" for c in prof["Cluster"]}\n'
    '    if len(real) >= 2:\n'
    '        tag[int(real["Cluster"].iloc[0])] = "Terlaris"\n'
    '        tag[int(real["Cluster"].iloc[-1])] = "Kurang laris"\n'
    '        for c in real["Cluster"].iloc[1:-1]:\n'
    '            tag[int(c)] = "Menengah"\n'
    '    prof["Julukan"] = [("Noise" if c == -1 else tag[int(c)] or "Satu cluster") for c in prof["Cluster"]]\n'
    '    return r, prof\n'
    '\n'
    '\n'
    'def cluster_cards(prof: pd.DataFrame) -> None:\n'
    '    rows = prof.to_dict("records")\n'
    '    for col, row in zip(st.columns(len(rows)), rows):\n'
    '        ch = row["Perubahan_13mg_%"]\n'
    '        color = V.cluster_color(row["Cluster"], config)\n'
    '        trend = "–" if not np.isfinite(ch) else f"{\'+\' if ch > 0 else \'−\'}{idn(abs(ch), 1)}%"\n'
    '        col.markdown(\n'
    '            f\'<div class="persona" style="--accent:{color}"><div class="bar" style="background:{color}"></div>\'\n'
    '            f\'<div class="kpi-label">{V.cluster_name(row["Cluster"])} · {row["Julukan"]}</div><h4>{idn(row["Jumlah_Produk"])} produk</h4>\'\n'
    '            f\'<div class="meta"><span class="chip">{row["Persona"]}</span> <span class="chip">{row["Tren"]}</span></div>\'\n'
    '            f\'<div class="kv"><span>Rata-rata / minggu</span><b>{idn(row["Rata2_Mingguan"], 1)} unit</b></div>\'\n'
    '            f\'<div class="kv"><span>Porsi volume</span><b>{idn(row["Porsi_Volume_%"], 1)}%</b></div>\'\n'
    '            f\'<div class="kv"><span>13 mg awal → akhir</span><b>{trend}</b></div></div>\', unsafe_allow_html=True)\n'
    '\n'
    '\n'
    'def product_table(W: np.ndarray, codes: np.ndarray, mask: np.ndarray) -> pd.DataFrame:\n'
    '    """Statistik per produk dalam satu cluster: rata-rata, variasi (CV), dan tren linear per minggu."""\n'
    '    G = W[mask]\n'
    '    mean = G.mean(axis=1)\n'
    '    cv = np.divide(G.std(axis=1), mean, out=np.full(len(G), np.nan), where=mean > 0)\n'
    '    t = np.arange(W.shape[1]) - (W.shape[1] - 1) / 2\n'
    '    slope = (G - mean[:, None]) @ t / (t @ t)\n'
    '    return pd.DataFrame({"Produk": codes[mask], "Rata-rata/minggu": mean, "Variasi (CV)": cv, "Tren/minggu": slope,\n'
    '                         "Pola": [list(map(float, g)) for g in G]}).sort_values("Rata-rata/minggu", ascending=False)\n'
    '\n'
    '\n'
    'def mean_cv_figure(W: np.ndarray, codes: np.ndarray, labels: np.ndarray):\n'
    '    """Tiap titik = 1 produk: rata-rata penjualan mingguan (log) vs kestabilan (CV)."""\n'
    '    import plotly.graph_objects as go\n'
    '    mean = W.mean(axis=1)\n'
    '    cv = np.divide(W.std(axis=1), mean, out=np.full(len(W), np.nan), where=mean > 0)\n'
    '    fig = go.Figure()\n'
    '    for c in sorted(set(labels.tolist())):\n'
    '        m = (labels == c) & (mean > 0)\n'
    '        fig.add_trace(go.Scatter(x=mean[m], y=cv[m], mode="markers", name=V.cluster_name(c), text=codes[m],\n'
    '                                 marker=dict(size=7, color=V.cluster_color(c, config), opacity=0.8, line=dict(width=0.5, color=config.PLOT_SURFACE)),\n'
    '                                 hovertemplate="%{text}<br>Rata-rata: %{x:.1f} unit/minggu<br>CV: %{y:.2f}<extra>" + V.cluster_name(c) + "</extra>"))\n'
    '    fig.update_xaxes(type="log", title_text="Rata-rata penjualan / minggu (skala log)")\n'
    '    fig.update_yaxes(title_text="Variasi (CV) · makin kecil makin stabil")\n'
    '    return V.style_figure(fig, None, 340, config)\n'
    '\n'
    '\n'
    'def product_series_figure(y: np.ndarray, name: str, color: str):\n'
    '    import plotly.graph_objects as go\n'
    '    fig = go.Figure(go.Scatter(x=np.arange(len(y)), y=y, mode="lines+markers", name=name, line=dict(color=color, width=2.4), marker=dict(size=5),\n'
    '                               hovertemplate="W%{x}: %{y:,.0f} unit<extra></extra>"))\n'
    '    fig.add_hline(y=float(np.mean(y)), line_dash="dot", line_color=config.PLOT_ACCENT, line_width=1.5,\n'
    '                  annotation_text=f"rata-rata {idn(np.mean(y), 1)}", annotation_font_color=config.PLOT_ACCENT)\n'
    '    fig.update_xaxes(title_text="Minggu", tickmode="array", tickvals=list(range(0, len(y), 8)), ticktext=[f"W{t}" for t in range(0, len(y), 8)])\n'
    '    fig.update_yaxes(title_text="Unit terjual", rangemode="tozero")\n'
    '    fig.update_layout(showlegend=False)\n'
    '    return V.style_figure(fig, None, 330, config)\n'
    '\n'
    '\n'
    'def drop_story(totals: np.ndarray) -> dict:\n'
    '    """Penurunan paling dalam setelah puncak, dan pemulihan di akhir periode (hitungan langsung dari total mingguan)."""\n'
    '    p = int(totals.argmax())\n'
    '    t = p + int(totals[p:].argmin())\n'
    '    return {"peak": p, "trough": t, "drop": 100 * (totals[t] - totals[p]) / totals[p], "last": float(totals[-1]), "recover": 100 * totals[-1] / totals[p]}\n'
    '\n'
    '\n'
    'def heading(number: str, title: str, subtitle: str) -> None:\n'
    '    st.markdown(f\'<div class="eyebrow">{number} · Ruang Sales</div>\', unsafe_allow_html=True)\n'
    '    st.title(title)\n'
    '    st.markdown(f\'<p class="page-sub">{subtitle}</p>\', unsafe_allow_html=True)\n'
    '\n'
    '\n'
    'def insight(title: str, color: str, body: str) -> str:\n'
    '    return f\'<div class="insight" style="border-left-color:{color}"><div class="t">{title}</div>{body}</div>\'\n'
    '\n'
    '\n'
    'def fmt(v, f="{:.3f}") -> str:\n'
    '    """Angka metrik dengan format Indonesia (3 desimal bawaan)."""\n'
    '    if v is None or (isinstance(v, float) and not np.isfinite(v)):\n'
    '        return "–"\n'
    '    return f.format(v).replace(",", "_").replace(".", ",").replace("_", ".")\n'
    '\n'
    '\n'
    'def quality(s) -> str:\n'
    '    if s is None or not np.isfinite(s):\n'
    '        return "Tidak terdefinisi"\n'
    '    return "Struktur kuat" if s > .7 else "Struktur wajar" if s > .5 else "Struktur lemah" if s > .25 else "Tanpa struktur berarti"\n'
    '\n'
    '\n'
    'def sidebar(page_objects: dict, current: str) -> None:\n'
    '    st.markdown(f\'<div class="brand"><div class="brand-icon">{icon("layers", 22)}</div><div><div class="brand-name">Ruang Sales</div>\'\n'
    '                \'<div class="brand-sub">Sales Analytics &amp; Time Series Clustering</div></div></div>\', unsafe_allow_html=True)\n'
    '    for i, (name, ic) in enumerate(zip(PAGES, ICONS), 1):\n'
    '        label = f"{i:02d}  {name}"\n'
    '        if name == current:\n'
    '            st.markdown(f\'<div class="nav-active">{icon(NAV_SVG[i - 1], 20)}{label}</div>\', unsafe_allow_html=True)\n'
    '        else:\n'
    '            st.page_link(page_objects[name], label=label, icon=f":material/{ic}:")\n'
    '    b = bundle()\n'
    '    if b is None:\n'
    '        return\n'
    '    key = active_key(b)\n'
    '    rep, algo, k, extra = key.split("|")\n'
    '    m = b["res"]["runs"][key]["metrics"]\n'
    '    rows = [("Representasi", REP_LABEL[rep].split(" (")[0]), ("Cluster", m["Cluster"]), ("Silhouette", fmt(m["Silhouette"])),\n'
    '            ("Balance", fmt(m["Balance"])), ("η² nol", fmt(m.get("Eta2_nol")))]\n'
    '    body = "".join(f\'<div class="kv"><span>{a}</span><b>{v}</b></div>\' for a, v in rows)\n'
    '    best = \'<span class="pill ok">terbaik</span>\' if key in b["res"]["best"].values() else ""\n'
    '    st.markdown(f\'<div class="side-note"><div class="side-head">{icon("sliders", 18)}<span>Model Aktif</span>{best}</div>\'\n'
    '                f\'<div class="side-model">{ALGO_LABEL[algo].split(" (")[0]}{" · eps p" + extra if algo == "dbscan" else ""}</div>{body}</div>\', unsafe_allow_html=True)\n'
    '    meta = b["res"]["metadata"]\n'
    '    st.markdown(f\'<div class="side-foot">{meta["n_products"]} produk · {meta["n_weeks"]} minggu<br>{len(b["res"]["runs"])} model (Colab) · FastDTW r={meta["distance"]["radius"]}</div>\', unsafe_allow_html=True)\n'
    '\n'
    '\n'
    'def upload_page() -> None:\n'
    '    """Ditampilkan bila folder hasil_model belum ada: unggah hasil_model.zip dari Colab."""\n'
    '    heading("00", "Hasil model belum ada.", "Jalankan notebook Colab <b>01_bangun_model</b>, lalu unggah <code>hasil_model.zip</code> di sini.")\n'
    '    up = st.file_uploader("hasil_model.zip", type="zip")\n'
    '    if up is not None:\n'
    '        with zipfile.ZipFile(io.BytesIO(up.getvalue())) as z:\n'
    '            names = [n for n in z.namelist() if n.startswith("hasil_model/")]\n'
    '            if "hasil_model/model_results.json" not in names:\n'
    '                st.error("Zip tidak berisi hasil_model/model_results.json. Pastikan file berasal dari notebook 01_bangun_model.")\n'
    '                return\n'
    '            z.extractall(config.PROJECT_ROOT, members=names)\n'
    '        st.success("Hasil model tersimpan. Memuat dashboard…")\n'
    '        st.rerun()\n'
    '\n'
    '\n'
    'def need_bundle() -> dict | None:\n'
    '    b = bundle()\n'
    '    if b is None:\n'
    '        upload_page()\n'
    '    return b\n'
    '\n'
    '\n'
    '# =======================================================================================\n'
    '# 01 · Ringkasan\n'
    '# =======================================================================================\n'
    'def page_overview() -> None:\n'
    '    b = need_bundle()\n'
    '    if b is None:\n'
    '        return\n'
    '    W, weekly, stats, res = b["W"], b["weekly"], b["stats"], b["res"]\n'
    '    totals = weekly["Total_Penjualan"].to_numpy()\n'
    '    sorted_totals = np.sort(W.sum(axis=1))[::-1]\n'
    '    top20 = 100 * sorted_totals[: round(.2 * len(sorted_totals))].sum() / sorted_totals.sum()\n'
    '    half = 100 * (totals[26:].sum() - totals[:26].sum()) / totals[:26].sum()\n'
    '    stats_html = "".join(\n'
    '        f\'<div class="hero-stat"><div class="ic{" gold" if gold else ""}">{icon(ic, 24)}</div><div><div class="v">{v}</div><div class="l">{l}</div></div></div>\'\n'
    '        for ic, v, l, gold in [("box", idn(len(W)), "Produk", True), ("layers", idn(W.sum()), "Unit terjual", False),\n'
    '                               ("bars", f"{idn(top20, 1)}%", "Volume dari 20% produk teratas", True), ("network", idn(len(res["runs"])), "Model dibangun di Colab", False)])\n'
    '    st.markdown(\'<div class="hero"><div><div class="eyebrow">Sales Transactions · W0–W51</div>\'\n'
    "                '<h1>Penjualan 52 Minggu,<br>dalam <em>Satu Pandangan</em>.</h1>'\n"
    "                '<p>Analisis pola penjualan mingguan dengan klasterisasi time series berbasis FastDTW pada tiga representasi data '\n"
    "                '(Data asli, Min-Max, Scale) dan enam algoritma.</p></div>'\n"
    '                f\'<div class="hero-stats">{stats_html}</div></div>\', unsafe_allow_html=True)\n'
    '    kpis([("Rata-rata / minggu", idn(totals.mean()), f"Puncak W{totals.argmax()}: {idn(totals.max())}", "calendar"),\n'
    '          ("Perubahan paruh 2 vs 1", f"{\'+\' if half > 0 else \'−\'}{idn(abs(half), 1)}%", "W26–W51 dibanding W0–W25", "trend_down" if half < 0 else "trend",\n'
    '           "var(--neg)" if half < 0 else "var(--emerald)"),\n'
    '          ("Produk aktif", f"{idn(weekly[\'Persen_Produk_Aktif\'].mean(), 1)}%", "rata-rata per minggu", "cube", "var(--gold)"),\n'
    '          ("Sel bernilai nol", f"{idn(100 * (W == 0).mean(), 1)}%", "dari 811 produk × 52 minggu", "filter", "var(--ink-2)")])\n'
    '    keys = {rep: k for rep, k in res["best"].items() if k}\n'
    '    if keys:\n'
    '        st.write("")\n'
    '        best_cards(b, keys)\n'
    '    st.write("")\n'
    '    story = drop_story(totals)\n'
    '    left, right = st.columns([1.55, 1], gap="medium")\n'
    '    with left, st.container(border=True):\n'
    '        section("Perjalanan penjualan mingguan", "Total semua produk dan rata-rata bergerak 4 minggu. Area emas = penurunan terdalam.", "trend")\n'
    '        fig = V.create_weekly_sales_figure(weekly, config)\n'
    '        if story["trough"] > story["peak"]:\n'
    '            fig.add_vrect(x0=story["peak"], x1=story["trough"], fillcolor=config.PLOT_ACCENT, opacity=0.14, line_width=0)\n'
    '        plot(fig, "ov_weekly")\n'
    '        st.markdown(f\'<div class="takeaway">Penjualan turun <b>{idn(abs(story["drop"]), 1)}%</b> dalam {story["trough"] - story["peak"]} minggu \'\n'
    '                    f\'(W{story["peak"]} → W{story["trough"]}), lalu pulih ke <b>{idn(story["last"])}</b> unit di akhir periode \'\n'
    '                    f\'(<b>{idn(story["recover"], 0)}%</b> dari puncak).</div>\', unsafe_allow_html=True)\n'
    '    with right, st.container(border=True):\n'
    '        section("Konsentrasi volume (Pareto)", "Produk diurutkan dari volume terbesar.", "bars")\n'
    '        plot(V.create_pareto_figure(stats, config), "ov_pareto")\n'
    '        st.markdown(f\'<div class="takeaway"><b>20%</b> produk teratas menyumbang <b>{idn(top20, 1)}%</b> volume.</div>\', unsafe_allow_html=True)\n'
    '\n'
    '    # ---- Profil klaster model aktif ----\n'
    '    key = active_key(b)\n'
    '    r, prof = cluster_story(b, key)\n'
    '    labels = r.labels\n'
    '    codes = b["codes"]\n'
    '    st.markdown(f\'<div class="band">{icon("network", 22)}<div><div class="band-t">Profil klaster</div>\'\n'
    '                f\'<div class="band-s">Model aktif: {key_label(key)}</div></div></div>\', unsafe_allow_html=True)\n'
    '    cluster_cards(prof)\n'
    '    st.write("")\n'
    '    a, c = st.columns(2, gap="medium")\n'
    '    with a, st.container(border=True):\n'
    '        section("Pola rata-rata mingguan tiap klaster", "Rata-rata penjualan asli anggota klaster.", "trend")\n'
    '        if len(r.cluster_ids):\n'
    '            centers = np.stack([W[labels == cid].mean(axis=0) for cid in r.cluster_ids])\n'
    '            plot(iv.centroid_overlay_figure(centers, r.cluster_ids, "mean", "Unit terjual (rata-rata)", config), "ov_pattern")\n'
    '    with c, st.container(border=True):\n'
    '        section("Rata-rata penjualan vs kestabilan", "Tiap titik = 1 produk.", "target")\n'
    '        plot(mean_cv_figure(W, codes, labels), "ov_meancv")\n'
    '\n'
    '    with st.container(border=True):\n'
    '        section("Produk teratas per klaster", "Pilih klaster, lalu pilih produk untuk melihat deret mingguannya.", "trophy")\n'
    '        names = {f\'{V.cluster_name(row["Cluster"])} · {row["Julukan"]}\': int(row["Cluster"]) for row in prof.to_dict("records")}\n'
    '        a, c = st.columns([1, 1.25], gap="medium")\n'
    '        with a:\n'
    '            pick = st.selectbox("Pilih klaster", list(names), key="ov_cluster")\n'
    '            cid = names.get(pick, next(iter(names.values())))\n'
    '            table = product_table(W, codes, labels == cid)\n'
    '            st.dataframe(table.head(15), hide_index=True, width="stretch", height=330,\n'
    '                         column_config={"Rata-rata/minggu": st.column_config.NumberColumn(format="%.2f"), "Variasi (CV)": st.column_config.NumberColumn(format="%.2f"),\n'
    '                                        "Tren/minggu": st.column_config.NumberColumn(format="%+.2f"), "Pola": st.column_config.LineChartColumn(width="small")})\n'
    '        with c:\n'
    '            options = table["Produk"].tolist()\n'
    '            product = st.selectbox("Lihat deret mingguan produk (bisa diketik untuk mencari)", options, key=f"ov_product_{cid}")\n'
    '            if product in options:\n'
    '                plot(product_series_figure(W[int(np.flatnonzero(codes == product)[0])], product, V.cluster_color(cid, config)), "ov_series")\n'
    '\n'
    '    # ---- Perbandingan representasi ----\n'
    '    with st.container(border=True):\n'
    '        section("Hasil klasterisasi", "Model terbaik pada tiga representasi data.", "layers")\n'
    '        rows = []\n'
    '        for rep, bkey in keys.items():\n'
    '            m = res["runs"][bkey]["metrics"]\n'
    '            rows.append({"Representasi": REP_LABEL[rep], "Model terbaik": key_label(bkey).split(" · ")[0] + " · " + key_label(bkey).split(" · ")[-1],\n'
    '                         "Silhouette": m["Silhouette"], "DBI": m["DBI_MDS"], "CHI": m["CHI_MDS"], "Balance": m["Balance"], "Cluster": m["Cluster"],\n'
    '                         "η² nol": m["Eta2_nol"], "Stabilitas": res["metadata"].get("stability", {}).get(rep, {}).get("status", "–")})\n'
    '        st.dataframe(pd.DataFrame(rows), hide_index=True, width="stretch",\n'
    '                     column_config={c2: st.column_config.NumberColumn(format="%.3f") for c2 in ["Silhouette", "DBI", "Balance", "η² nol"]} | {"CHI": st.column_config.NumberColumn(format="%.0f")})\n'
    '    a, c = st.columns(2, gap="medium")\n'
    '    with a, st.container(border=True):\n'
    '        section("10 produk dengan volume tertinggi", "Warna = klaster model aktif.", "trophy")\n'
    '        plot(V.create_top_products_figure(stats, 10, config, dict(zip(codes, labels))), "ov_top")\n'
    '    with c, st.container(border=True):\n'
    '        section("Produk aktif & minggu tanpa penjualan", "", "zero")\n'
    '        plot(V.create_active_products_figure(weekly, config), "ov_active")\n'
    '        plot(V.create_zero_weeks_figure(stats, config), "ov_zero")\n'
    '\n'
    '\n'
    '# =======================================================================================\n'
    '# 02 · Eksplorasi data\n'
    '# =======================================================================================\n'
    'def page_exploration() -> None:\n'
    '    b = need_bundle()\n'
    '    if b is None:\n'
    '        return\n'
    '    heading("02", "Kenali data sebelum mengelompokkan.", "Detail per produk pada tiga representasi, hubungan aktivitas–volume, dan peta panas seluruh produk.")\n'
    '    key = active_key(b)\n'
    '    labels = np.asarray(b["res"]["runs"][key]["labels"])\n'
    '    codes = b["codes"].tolist()\n'
    '    with st.container(border=True):\n'
    '        h1, h2 = st.columns([3, 1.4])\n'
    '        h1.subheader(f"Peta panas {len(codes)} produk × 52 minggu")\n'
    '        h1.caption(f"Dikelompokkan menurut model aktif ({key_label(key)}), lalu volume terbesar. Putih = 0 unit.")\n'
    '        mode = h2.segmented_control("Warna", ["Min-Max", "Scale (z-score)", "Log data asli"], default="Min-Max", key="ex_heat") or "Min-Max"\n'
    '        heat = {"Min-Max": "minmax", "Scale (z-score)": "zscore"}.get(mode, "log")\n'
    '        plot(V.create_sales_heatmap_figure(b["W"], b["X"]["zscore" if heat == "zscore" else "minmax"], labels, b["codes"], heat, config), "ex_heatmap")\n'
    '    left, right = st.columns([1.55, 1], gap="medium")\n'
    '    with left, st.container(border=True):\n'
    '        c1, c2, c3 = st.columns([1, 1, 1.3])\n'
    '        prod = c1.selectbox("Produk", codes, key="ex_prod")\n'
    '        cmp = c2.selectbox("Bandingkan", ["—", *codes], key="ex_cmp")\n'
    '        scale_name = c3.segmented_control("Skala", ["Data asli", "Min-Max", "Scale"], default="Data asli", key="ex_scale") or "Data asli"\n'
    '        scale = {"Data asli": "raw", "Min-Max": "minmax", "Scale": "zscore"}[scale_name]\n'
    '        figs = V.create_exploration_figures(b["stats"], b["W"], prod, config, labels=labels, compare_product=None if cmp == "—" else cmp,\n'
    '                                            scale=scale, X_scaled=b["X"].get(scale) if scale != "raw" else None)\n'
    '        section(f"Deret mingguan · {prod}", "Garis titik = median cluster produk ini sebagai konteks.", "trend")\n'
    '        plot(figs["product_detail"], "ex_series")\n'
    '    with right, st.container(border=True):\n'
    '        section("Label di setiap representasi", "Cluster produk ini pada model terbaik tiap representasi.", "layers")\n'
    '        i = codes.index(prod)\n'
    '        rows = []\n'
    '        for rep, bk in b["res"]["best"].items():\n'
    '            if bk:\n'
    '                run = b["res"]["runs"][bk]\n'
    '                rows.append({"Representasi": REP_LABEL[rep], "Model": key_label(bk).split(" · ")[0], "Cluster": V.cluster_name(run["labels"][i]),\n'
    '                             "Silhouette": run["silhouette"][i]})\n'
    '        st.dataframe(pd.DataFrame(rows), hide_index=True, width="stretch", column_config={"Silhouette": st.column_config.NumberColumn(format="%.3f")})\n'
    '        row = b["stats"].iloc[i]\n'
    '        st.metric("Total volume", f"{row[\'Total volume\']:,.0f}")\n'
    '        st.metric("Minggu nol", f"{int(row[\'Minggu nol\'])} / 52")\n'
    '    a, c = st.columns(2, gap="medium")\n'
    '    with a, st.container(border=True):\n'
    '        section("Aktivitas vs volume", "Sumbu Y skala log; warna = cluster model aktif.", "map")\n'
    '        plot(figs["activity_vs_volume"], "ex_scatter")\n'
    '    with c, st.container(border=True):\n'
    '        section("Distribusi total volume", "Bin logaritmik: distribusi sangat menceng ke kanan.", "bars")\n'
    '        plot(figs["sales_distribution"], "ex_hist")\n'
    '\n'
    '\n'
    '# =======================================================================================\n'
    '# 03 · Eksplorasi jarak\n'
    '# =======================================================================================\n'
    'def _fastdtw(a: np.ndarray, b: np.ndarray, radius: int):\n'
    '    """FastDTW (numba) dengan jalur; jika numba tidak ada, CDTW pita = radius sebagai aproksimasi."""\n'
    '    try:\n'
    '        from scripts.distance import fastdtw_numba\n'
    '        d, pi, pj = fastdtw_numba(np.ascontiguousarray(a, dtype=np.float64), np.ascontiguousarray(b, dtype=np.float64), int(radius), True)\n'
    '        return float(d), np.c_[pi, pj], "FastDTW (numba)"\n'
    '    except Exception:\n'
    '        d, path, _ = ix.dtw_pair(a, b, int(radius))\n'
    '        return d, path, f"CDTW pita {radius} (numba tidak tersedia)"\n'
    '\n'
    '\n'
    'def page_distance() -> None:\n'
    '    b = need_bundle()\n'
    '    if b is None:\n'
    '        return\n'
    '    heading("03", "Seberapa mirip dua produk?", "Euclidean vs DTW vs FastDTW pada representasi pilihan, lengkap dengan jalur penyejajaran minggu.")\n'
    '    codes = b["codes"].tolist()\n'
    '    radius = b["res"]["metadata"]["distance"]["radius"]\n'
    '    with st.container(border=True):\n'
    '        c1, c2, c3 = st.columns(3)\n'
    '        rep = c1.selectbox("Representasi", list(b["X"]), format_func=REP_LABEL.get, key="dj_rep")\n'
    '        pa = c2.selectbox("Produk A", codes, index=0, key="dj_a")\n'
    '        pb = c3.selectbox("Produk B", codes, index=1, key="dj_b")\n'
    '    X = b["X"][rep]\n'
    '    ia, ib = codes.index(pa), codes.index(pb)\n'
    '    eu = float(np.sqrt(((X[ia] - X[ib]) ** 2).sum()))\n'
    '    d_dtw, path_dtw, cost = ix.dtw_pair(X[ia], X[ib], None)\n'
    '    d_fast, path_fast, engine = _fastdtw(X[ia], X[ib], radius)\n'
    '    kpis([("Euclidean", fmt(eu), "minggu dipasangkan 1-ke-1", "grid", "var(--ink-2)"), ("DTW penuh", fmt(d_dtw), "jalur optimal eksak", "pulse"),\n'
    '          (f"FastDTW r = {radius}", fmt(d_fast), engine, "wave", "var(--gold)"), ("Pasangan bergeser", f"{int((path_fast[:, 0] != path_fast[:, 1]).sum())} / {len(path_fast)}", "langkah jalur di luar diagonal", "trend", "var(--emerald)")])\n'
    '    left, right = st.columns([1.3, 1], gap="medium")\n'
    '    with left, st.container(border=True):\n'
    '        section("Week alignment · FastDTW", "Garis abu-abu = minggu yang disejajarkan; miring = pergeseran waktu.", "wave")\n'
    '        plot(iv.dtw_alignment_figure(X[ia], X[ib], path_fast, pa, pb, Y_TITLE[rep], config), "dj_align")\n'
    '    with right, st.container(border=True):\n'
    '        section("Matriks biaya & jalur", "Warna = biaya kumulatif DTW penuh; merah = jalur FastDTW.", "grid")\n'
    '        plot(iv.dtw_cost_figure(cost, path_fast, None, pa, pb, config), "dj_cost")\n'
    '    with st.container(border=True):\n'
    '        section(f"10 produk paling mirip dengan {pa}", "FastDTW ke seluruh produk pada representasi terpilih.", "target")\n'
    '        dist = np.array([_fastdtw(X[ia], X[j], radius)[0] for j in range(len(X))])\n'
    '        frame = pd.DataFrame({"Produk": codes, "FastDTW": dist, "Euclidean": np.sqrt(((X - X[ia]) ** 2).sum(axis=1)),\n'
    '                              "Total volume": b["W"].sum(axis=1), "Pola": [list(map(float, r)) for r in X]}).drop(index=ia).nsmallest(10, "FastDTW")\n'
    '        st.dataframe(frame, hide_index=True, width="stretch", column_config={"FastDTW": st.column_config.NumberColumn(format="%.3f"),\n'
    '                     "Euclidean": st.column_config.NumberColumn(format="%.3f"), "Pola": st.column_config.LineChartColumn(width="medium")})\n'
    '\n'
    '\n'
    '# =======================================================================================\n'
    '# 04 · Klasterisasi (pilih → Tampilkan)\n'
    '# =======================================================================================\n'
    'def _apply_preset() -> None:\n'
    '    b = bundle()\n'
    '    choice = st.session_state.get("cl_preset")\n'
    '    if not b or not choice or choice == "Kustom":\n'
    '        return\n'
    '    rep, algo, k, extra = b["res"]["best"][choice].split("|")\n'
    '    st.session_state.update(cl_rep=rep, cl_algo=algo)\n'
    '    if algo == "dbscan":\n'
    '        st.session_state["cl_eps"] = int(extra)\n'
    '    else:\n'
    '        st.session_state["cl_k"] = int(k)\n'
    '    if algo == "agglomerative":\n'
    '        st.session_state["cl_link"] = extra\n'
    '\n'
    '\n'
    'def page_clustering() -> None:\n'
    '    b = need_bundle()\n'
    '    if b is None:\n'
    '        return\n'
    '    heading("04", "Hasil klasterisasi.", "Pilih representasi, algoritma, dan parameter, lalu tekan <b>Tampilkan hasil</b>. Semua model sudah dibangun di Colab dengan FastDTW.")\n'
    '    cfg = b["res"]["metadata"]["config"]\n'
    '    best = b["res"]["best"]\n'
    '    with st.container(border=True):\n'
    '        section("Kontrol parameter", "Preset mengisi pilihan dengan model terbaik; ubah sesuai kebutuhan lalu tekan Tampilkan hasil.", "sliders")\n'
    '        st.selectbox("Preset", ["Kustom", *[r for r, k in best.items() if k]], key="cl_preset", on_change=_apply_preset,\n'
    '                     format_func=lambda r: r if r == "Kustom" else f"Model terbaik · {REP_LABEL[r]} ({key_label(best[r]).split(\' · \', 2)[0]})")\n'
    '        c1, c2, c3, c4 = st.columns(4)\n'
    '        rep = c1.selectbox("Representasi data", list(b["res"]["spaces"]), format_func=REP_LABEL.get, key="cl_rep",\n'
    '                           help="Data asli: volume ikut menentukan. Min-Max: bentuk 0–1. Scale: bentuk relatif terhadap rata-rata & simpangan produk.")\n'
    '        algo = c2.selectbox("Algoritma", ms.ALGORITHMS, format_func=ALGO_LABEL.get, key="cl_algo")\n'
    '        if algo == "dbscan":\n'
    '            extra = c3.selectbox("eps (persentil jarak tetangga)", cfg["DBSCAN_EPS_PERCENTILES"], index=len(cfg["DBSCAN_EPS_PERCENTILES"]) // 2,\n'
    '                                 format_func=lambda p: f"persentil {p}", key="cl_eps")\n'
    '            key = ms.run_key(rep, algo, 0, extra)\n'
    '        else:\n'
    '            k = c3.select_slider("Jumlah cluster K", cfg["K_GRID"], key="cl_k")\n'
    '            extra = c4.selectbox("Linkage", cfg["LINKAGES"], key="cl_link") if algo == "agglomerative" else None\n'
    '            key = ms.run_key(rep, algo, k, extra)\n'
    '        st.caption({"kmeans": "Centroid = barycenter DBA (rata-rata yang menghormati pergeseran waktu).",\n'
    '                    "kmedoids": "Centroid = medoid (produk nyata), PAM BUILD + SWAP.",\n'
    '                    "agglomerative": "Tidak punya pusat bawaan; medoid tiap cluster + dendrogram.",\n'
    '                    "dbscan": f"min_samples = {cfg[\'DBSCAN_MIN_SAMPLES\']}; produk di wilayah jarang = noise.",\n'
    '                    "fcm": f"Fuzzy m = {cfg[\'FUZZINESS\']}; centroid DBA berbobot keanggotaan.",\n'
    '                    "fcmdd": f"Fuzzy m = {cfg[\'FUZZINESS\']}; centroid = medoid berbobot keanggotaan."}[algo])\n'
    '    b1, b2 = st.columns([1, 3])\n'
    '    if b1.button("Tampilkan hasil", type="primary", icon=":material/play_arrow:", width="stretch"):\n'
    '        st.session_state["shown_key"] = key\n'
    '        st.session_state["active_key"] = key\n'
    '    shown = st.session_state.get("shown_key")\n'
    '    if shown and shown != key:\n'
    '        b2.warning("Pilihan berubah sejak hasil terakhir. Tekan **Tampilkan hasil** untuk memperbarui.", icon=":material/sync_problem:")\n'
    '    if not shown:\n'
    '        st.markdown(\'<div class="callout"><b>Belum ada hasil.</b> Pilih skenario di atas lalu tekan <b>Tampilkan hasil</b>. \'\n'
    "                    'Pilih <i>Preset</i> untuk langsung melihat model terbaik tiap representasi.</div>', unsafe_allow_html=True)\n"
    '        return\n'
    '    render_result(b, shown)\n'
    '\n'
    '\n'
    'def render_result(b: dict, key: str) -> None:\n'
    '    rep, algo, k, extra = key.split("|")\n'
    '    r = result(b, key)\n'
    '    space = b["res"]["spaces"][rep]\n'
    '    X, W, codes = b["X"][rep], b["W"], b["codes"]\n'
    '    m = r.metrics\n'
    '    kpis([("Silhouette", fmt(m["Silhouette"]), f"{quality(m[\'Silhouette\'])} · pemisahan antar-cluster (↑)", "target", "var(--cyan)"),\n'
    '          ("DBI (MDS)", fmt(m["DBI_MDS"]), "Davies–Bouldin: makin kecil makin terpisah (↓)", "scale", "var(--gold)"),\n'
    '          ("CHI (MDS)", idn(m["CHI_MDS"]), "Calinski–Harabasz: makin besar makin padat (↑)", "bars", "var(--emerald)"),\n'
    '          ("Balance", fmt(m["Balance"]), "1 = ukuran cluster seimbang", "layers", "var(--cyan)"),\n'
    '          ("Jumlah cluster", str(int(m["Cluster"])), f"coverage {idn(100 * m[\'Coverage\'], 1)}%" + (f" · {int(m[\'Noise\'])} noise" if m["Noise"] else ""), "network", "var(--gold)"),\n'
    '          ("η² minggu nol", fmt(m["Eta2_nol"]), "tinggi = cluster dipisah frekuensi nol", "zero", "var(--neg)")])\n'
    '    is_best = key in b["res"]["best"].values()\n'
    '    st.caption(f"{key_label(key)} · FastDTW · pusat: {ix.CENTER_KIND[r.center_kind]}" + (" · ★ model terbaik representasi ini" if is_best else ""))\n'
    '    for note in r.notes:\n'
    '        st.caption(f"• {note}")\n'
    '    if m["Cluster"] < 2:\n'
    '        st.warning("Hanya terbentuk satu cluster" + (" padat; sisanya noise. Pilih persentil eps lain." if algo == "dbscan" else "."), icon=":material/info:")\n'
    '    tabs = st.tabs(["Peta cluster & centroid", "Pola centroid", "Kualitas & diagnostik", "Profil & interpretasi", "Peta panas anggota"])\n'
    '    with tabs[0]:\n'
    '        c1, c2, _ = st.columns([1.2, 1, 2])\n'
    '        shape = c1.segmented_control("Batas cluster", ["hull", "ellipse", "none"], default="hull", key="fv_shape",\n'
    '                                     format_func={"hull": "Convex hull", "ellipse": "Elips 95%", "none": "Tanpa"}.get) or "hull"\n'
    '        spider = c2.toggle("Garis ke centroid", key="fv_spider")\n'
    '        st.caption("Gaya `fviz_cluster`: koordinat MDS klasik dari matriks FastDTW. ◆ = centroid (DBA diproyeksikan formula Gower), ★ = medoid / produk representatif.")\n'
    '        xy = np.array(space["xy"], dtype=float)\n'
    '        plot(iv.fviz_cluster_figure(xy, np.array(space["explained"], dtype=float), r.labels, codes, r.center_xy, r.cluster_ids, r.medoids,\n'
    '                                    r.center_kind, shape, spider, config, "Dim"), "fv_map")\n'
    '        a, c = st.columns([1.6, 1], gap="medium")\n'
    '        with a, st.container(border=True):\n'
    '            section("Semua centroid", "Bentuk pola tiap cluster dalam satu grafik.", "trend")\n'
    '            if len(r.cluster_ids):\n'
    '                plot(iv.centroid_overlay_figure(r.centers, r.cluster_ids, r.center_kind, Y_TITLE[rep], config), "fv_overlay")\n'
    '        with c, st.container(border=True):\n'
    '            section("Jarak antar-centroid", "FastDTW/DTW; makin jauh makin terpisah.", "grid")\n'
    '            if len(r.cluster_ids) >= 2:\n'
    '                plot(iv.center_distance_figure(r.center_dm, r.cluster_ids, config), "fv_cdm")\n'
    '    with tabs[1]:\n'
    '        view = st.segmented_control("Skala", ["rep", "raw"], default="rep", key="cs_view",\n'
    '                                    format_func={"rep": f"Representasi ({REP_LABEL[rep]})", "raw": "Penjualan asli"}.get) or "rep"\n'
    '        if len(r.cluster_ids):\n'
    '            if view == "raw":\n'
    '                centers = np.stack([W[r.labels == c].mean(axis=0) for c in r.cluster_ids])\n'
    '                plot(iv.centroid_series_figure(W, r.labels, centers, r.cluster_ids, r.medoids, codes, "mean", "Penjualan asli", config), "cs_raw")\n'
    '            else:\n'
    '                plot(iv.centroid_series_figure(X, r.labels, r.centers, r.cluster_ids, r.medoids, codes, r.center_kind, Y_TITLE[rep], config), "cs_rep")\n'
    '            for col, c, med in zip(st.columns(len(r.cluster_ids)), r.cluster_ids, r.medoids):\n'
    '                mask = r.labels == c\n'
    '                col.markdown(f\'<div class="persona" style="min-height:0;text-align:center"><div class="bar" style="background:{V.cluster_color(c, config)}"></div>\'\n'
    '                             f\'<div class="kpi-label">Produk representatif {V.cluster_name(c)}</div><h4 style="font-size:24px">{codes[med]}</h4>\'\n'
    '                             f\'<span class="chip">{mask.sum()} produk</span> <span class="chip">Total {W[mask].sum():,.0f}</span></div>\', unsafe_allow_html=True)\n'
    '    with tabs[2]:\n'
    '        a, c = st.columns(2, gap="medium")\n'
    '        with a, st.container(border=True):\n'
    '            section("Plot silhouette", "Gaya fviz_silhouette; batang < 0 = mungkin salah cluster.", "target")\n'
    '            if np.isfinite(m["Silhouette"]):\n'
    '                plot(iv.fviz_silhouette_figure(r.silhouette, r.labels, codes, config), "q_sil")\n'
    '                st.caption(f"{100 * np.nanmean(r.silhouette[r.labels != -1] < 0):.1f}% produk bernilai negatif (lebih dekat ke cluster lain).")\n'
    '        with c, st.container(border=True):\n'
    '            section("Jarak anggota ke pusat", "Ekor panjang = anggota pinggiran.", "scale")\n'
    '            plot(iv.distance_to_center_figure(r.dist_to_center, r.labels, codes, config), "q_d2c")\n'
    '        if algo == "agglomerative":\n'
    '            with st.container(border=True):\n'
    '                section(f"Dendrogram · linkage {extra}", "Cabang diwarnai per cluster; garis merah = posisi potong.", "network")\n'
    '                plot(iv.fviz_dend_figure(r.linkage, r.labels, int(k), config), "q_dend")\n'
    '        if algo == "dbscan":\n'
    '            with st.container(border=True):\n'
    '                section("kNN-distance plot", "eps ideal di lutut kurva.", "trend")\n'
    '                plot(iv.knn_distance_figure(np.array(space["kdist"], dtype=float), r.eps, b["res"]["metadata"]["config"]["DBSCAN_MIN_SAMPLES"], config), "q_knn")\n'
    '        if r.membership is not None:\n'
    '            with st.container(border=True):\n'
    '                section("Keanggotaan fuzzy", "", "layers")\n'
    '                st.caption(f"FPC {fmt(m.get(\'FPC\'))} · entropi partisi {fmt(m.get(\'Entropi_partisi\'))}. Mendekati 0,5 = produk di perbatasan.")\n'
    '                plot(iv.membership_stack_figure(r.membership, r.labels, codes, config), "q_mem")\n'
    '        if algo != "dbscan":\n'
    '            with st.container(border=True):\n'
    '                section("Berapa K yang tepat?", "Metode siku + rata-rata silhouette dari model K = 2…6 yang sudah dibangun.", "bars")\n'
    '                rows = [{"k": kk, **b["res"]["runs"][ms.run_key(rep, algo, kk, extra)]["metrics"]} for kk in b["res"]["metadata"]["config"]["K_GRID"]]\n'
    '                rows = [{**row, **{c: (np.nan if v is None else v) for c, v in row.items()}} for row in rows]\n'
    '                plot(iv.fviz_nbclust_figure(rows, int(k), config), "q_nb")\n'
    '    with tabs[3]:\n'
    '        prof = pd.DataFrame(ix.cluster_profile_rows(r, W, codes))\n'
    '        prof.insert(0, "Nama", prof["Cluster"].map(V.cluster_name))\n'
    '        section("Ringkasan anggota dan produk representatif", "", "table")\n'
    '        st.dataframe(prof[["Nama", "Persona", "Jumlah_Produk", "Proporsi_%", "Total_Penjualan", "Porsi_Volume_%", "Produk_Representatif", "Rata2_Jarak_ke_Pusat"]],\n'
    '                     hide_index=True, width="stretch", column_config={"Porsi_Volume_%": st.column_config.ProgressColumn("Porsi volume", format="%.1f%%", min_value=0, max_value=100),\n'
    '                                                                       "Proporsi_%": st.column_config.NumberColumn(format="%.1f"), "Rata2_Jarak_ke_Pusat": st.column_config.NumberColumn(format="%.3f")})\n'
    '        section("Profilisasi deskriptif", "Dihitung dari penjualan asli.", "pulse")\n'
    '        st.dataframe(prof[["Nama", "Rata2_Mingguan", "Median_Minggu_Nol", "CV_Profil", "Perubahan_13mg_%", "Tren"]], hide_index=True, width="stretch",\n'
    '                     column_config={"Rata2_Mingguan": st.column_config.NumberColumn(format="%.2f"), "CV_Profil": st.column_config.NumberColumn(format="%.3f"),\n'
    '                                    "Perubahan_13mg_%": st.column_config.NumberColumn("13 mg awal → akhir", format="%+.1f%%")})\n'
    '        section("Interpretasi dan rekomendasi bisnis", "Deskriptif dari data historis; validasi dengan konteks bisnis.", "info")\n'
    '        for row in prof.to_dict("records"):\n'
    '            ch = row["Perubahan_13mg_%"]\n'
    '            st.markdown(f\'<div class="insight" style="min-height:0;margin-bottom:10px;border-left-color:{V.cluster_color(row["Cluster"], config)}">\'\n'
    '                        f\'<b>{row["Nama"]} · {row["Tren"]}</b> &nbsp;<span class="chip">{row["Persona"]}</span><br><span class="kpi-detail">{row["Jumlah_Produk"]} produk · \'\n'
    '                        f\'13 minggu awal → akhir: {"–" if not np.isfinite(ch) else f"{ch:+.1f}%"} · CV {row["CV_Profil"]:.3f}</span><br>{row["Rekomendasi"]}</div>\', unsafe_allow_html=True)\n'
    '        members = pd.DataFrame({"Product_Code": codes, "Cluster": [V.cluster_name(c) for c in r.labels], "Total": W.sum(axis=1),\n'
    '                                "Minggu_nol": (W == 0).sum(axis=1), "Silhouette": r.silhouette, "Jarak_ke_pusat": r.dist_to_center})\n'
    '        if r.membership is not None:\n'
    '            members["Keanggotaan"] = r.membership.max(axis=1)\n'
    '        pick = st.selectbox("Lihat anggota", sorted(members["Cluster"].unique()), key="pf_pick")\n'
    '        st.dataframe(members[members["Cluster"] == pick].sort_values("Total", ascending=False), hide_index=True, width="stretch", height=320,\n'
    '                     column_config={c: st.column_config.NumberColumn(format="%.3f") for c in ["Silhouette", "Jarak_ke_pusat", "Keanggotaan"]})\n'
    '        st.download_button("Unduh label (CSV)", members.to_csv(index=False).encode("utf-8-sig"), f"label_{key.replace(\'|\', \'_\')}.csv", "text/csv", icon=":material/download:")\n'
    '    with tabs[4]:\n'
    '        st.caption("Per cluster, diurutkan dari produk paling dekat ke pusat (atas) ke paling jauh (bawah).")\n'
    '        plot(iv.cluster_heatmap_figure(X, r.labels, r.dist_to_center, codes, config), "hm")\n'
    '\n'
    '\n'
    '# =======================================================================================\n'
    '# 05 · Rekomendasi\n'
    '# =======================================================================================\n'
    'def page_recommendation() -> None:\n'
    '    b = need_bundle()\n'
    '    if b is None:\n'
    '        return\n'
    '    heading("05", "Model mana yang paling cocok?", "Model terbaik tiap representasi dipilih di Colab: lolos guardrail lalu peringkat rata-rata Silhouette↑, DBI↓, CHI↑.")\n'
    '    res, meta = b["res"], b["res"]["metadata"]\n'
    '    cfg = meta["config"]\n'
    '    st.caption(f"Guardrail: minimal {cfg[\'MIN_CLUSTERS\']} cluster · Balance ≥ {cfg[\'MIN_BALANCE\']} · noise ≤ {int(100 * cfg[\'MAX_NOISE_SHARE\'])}% produk.")\n'
    '    t1, t2, t3 = st.tabs(["Terbaik per representasi", "Peringkat per representasi", "Semua model"])\n'
    '    with t1:\n'
    '        best_cards(b, {rep: k for rep, k in res["best"].items() if k})\n'
    '        st.write("")\n'
    '        rows = []\n'
    '        for rep, key in res["best"].items():\n'
    '            if key:\n'
    '                m = res["runs"][key]["metrics"]\n'
    '                rows.append({"Representasi": REP_LABEL[rep], "Model terbaik": key_label(key), **{c: m[c] for c in ["Silhouette", "DBI_MDS", "CHI_MDS", "Balance", "Cluster", "Eta2_nol"]},\n'
    '                             "Stabilitas seed": meta.get("stability", {}).get(rep, {}).get("status", "–")})\n'
    '        st.dataframe(pd.DataFrame(rows), hide_index=True, width="stretch",\n'
    '                     column_config={c: st.column_config.NumberColumn(format="%.3f") for c in ["Silhouette", "DBI_MDS", "Balance", "Eta2_nol"]} | {"CHI_MDS": st.column_config.NumberColumn(format="%.0f")})\n'
    '        keys = {rep: k for rep, k in res["best"].items() if k}\n'
    '        if st.button("Buka model terbaik representasi default di Klasterisasi", icon=":material/arrow_forward:"):\n'
    '            st.session_state["shown_key"] = st.session_state["active_key"] = res["best"].get(config.DEFAULT_REPRESENTATION) or next(iter(keys.values()))\n'
    '            st.switch_page(st.session_state["_pages"]["Klasterisasi"])\n'
    '    with t2:\n'
    '        rep = st.segmented_control("Representasi", list(res["spaces"]), default=config.DEFAULT_REPRESENTATION, format_func=REP_LABEL.get, key="rc_rep") or config.DEFAULT_REPRESENTATION\n'
    '        table = b["table"][b["table"]["Kunci"].str.startswith(rep + "|")].copy()\n'
    '        _, ranked = ms.select_best(res["runs"], rep, meta["n_products"], type("C", (), cfg)())\n'
    '        rank_map = dict(zip(ranked["Kunci"], ranked.get("Skor_Ranking", pd.Series(dtype=float)))) if "Skor_Ranking" in ranked else {}\n'
    '        table["Skor ranking"] = table["Kunci"].map(rank_map)\n'
    '        table["Lolos guardrail"] = table["Kunci"].isin(rank_map)\n'
    '        table = table.sort_values(["Lolos guardrail", "Skor ranking", "Silhouette"], ascending=[False, True, False])\n'
    '        st.dataframe(table[["Algoritma", "K", "Parameter", "Lolos guardrail", "Skor ranking", "Silhouette", "DBI_MDS", "CHI_MDS", "Balance", "Cluster", "Noise", "Eta2_nol", "Terbaik"]],\n'
    '                     hide_index=True, width="stretch", height=520,\n'
    '                     column_config={"Silhouette": st.column_config.ProgressColumn(format="%.3f", min_value=-1, max_value=1),\n'
    '                                    "Balance": st.column_config.ProgressColumn(format="%.3f", min_value=0, max_value=1),\n'
    '                                    "Skor ranking": st.column_config.NumberColumn(format="%.2f"), "DBI_MDS": st.column_config.NumberColumn(format="%.3f"),\n'
    '                                    "CHI_MDS": st.column_config.NumberColumn(format="%.0f"), "Eta2_nol": st.column_config.NumberColumn("η² nol", format="%.3f")})\n'
    '    with t3:\n'
    '        st.dataframe(b["table"].drop(columns=["Kunci"]), hide_index=True, width="stretch", height=520)\n'
    '        st.download_button("Unduh metrik semua model (CSV)", b["table"].to_csv(index=False).encode("utf-8-sig"), "metrik_semua_model.csv", "text/csv", icon=":material/download:")\n'
    '\n'
    '\n'
    '# =======================================================================================\n'
    '# 06 · Metodologi\n'
    '# =======================================================================================\n'
    'def page_methodology() -> None:\n'
    '    b = need_bundle()\n'
    '    if b is None:\n'
    '        return\n'
    '    heading("06", "Dari data mentah menjadi pola.", "Alur singkat pembangunan model dan cara membaca hasilnya.")\n'
    '    res = b["res"]\n'
    '    meta, cfg = res["metadata"], res["metadata"]["config"]\n'
    '    n_alg = len({k.split("|")[1] for k in res["runs"]})\n'
    '    steps = [("table", "Data", f"{idn(meta[\'n_products\'])} produk × {meta[\'n_weeks\']} minggu"),\n'
    '             ("layers", "Representasi", "Data asli · Min-Max · Scale"),\n'
    '             ("trend", "Jarak", f"FastDTW radius {meta[\'distance\'][\'radius\']}"),\n'
    '             ("network", "Klasterisasi", f"{n_alg} algoritma · K {min(cfg[\'K_GRID\'])}–{max(cfg[\'K_GRID\'])}"),\n'
    '             ("trophy", "Pemilihan", f"{idn(len(res[\'runs\']))} model → 1 terbaik per representasi")]\n'
    '    st.markdown(\'<div class="flow">\' + "".join(\n'
    '        f\'<div class="flow-step"><div class="flow-ic">{icon(ic, 22)}</div><div class="flow-n">0{i}</div><div class="flow-t">{t}</div><div class="flow-d">{d}</div></div>\'\n'
    '        for i, (ic, t, d) in enumerate(steps, 1)) + "</div>", unsafe_allow_html=True)\n'
    '\n'
    '    a, c = st.columns([1.5, 1], gap="medium")\n'
    '    with a, st.container(border=True):\n'
    '        section("Cara membaca metrik", "", "target")\n'
    '        metrics = [("Silhouette", "↑", "Pemisahan antar-klaster (−1 sampai 1)."), ("DBI", "↓", "Tumpang-tindih klaster; kecil = terpisah."),\n'
    '                   ("CHI", "↑", "Kepadatan klaster; besar = padat."), ("Balance", "↑", "Keseimbangan ukuran klaster (0–1)."),\n'
    '                   ("η² minggu nol", "↓", "Tinggi = klaster hanya memisah produk jarang laku.")]\n'
    '        st.markdown(\'<div class="mgrid">\' + "".join(\n'
    '            f\'<div class="mcard"><div class="mhead"><b>{n}</b><span class="{"up" if d == "↑" else "down"}">{d}</span></div><div class="mdesc">{t}</div></div>\'\n'
    '            for n, d, t in metrics) + "</div>", unsafe_allow_html=True)\n'
    '    with c, st.container(border=True):\n'
    '        section("Aturan memilih model terbaik", "", "filter")\n'
    '        rules = [("1", f"Minimal {cfg[\'MIN_CLUSTERS\']} klaster"), ("2", f"Balance ≥ {idn(cfg[\'MIN_BALANCE\'], 1)}"),\n'
    '                 ("3", f"Noise ≤ {idn(100 * cfg[\'MAX_NOISE_SHARE\'], 0)}% produk"), ("4", "Peringkat rata-rata Silhouette ↑, DBI ↓, CHI ↑")]\n'
    '        st.markdown(\'<div class="rules">\' + "".join(f\'<div class="rule"><span>{n}</span><div>{t}</div></div>\' for n, t in rules) + "</div>", unsafe_allow_html=True)\n'
    '\n'
    '    with st.container(border=True):\n'
    '        section("Model terbaik per representasi", "", "trophy")\n'
    '        best_cards(b, {rep: k for rep, k in res["best"].items() if k})\n'
    '\n'
    '    with st.expander("Detail teknis & unduhan", icon=":material/settings:"):\n'
    '        st.json({"dibangun_utc": meta["created_at_utc"], "durasi_detik": meta["runtime_s"], "jarak": meta["distance"], "konfigurasi": cfg,\n'
    '                 "model_terbaik": res["best"], "stabilitas": meta.get("stability"), "versi": meta["versions"]}, expanded=False)\n'
    '        files = [p for p in sorted(config.RESULTS_DIR.iterdir()) if p.is_file()]\n'
    '        buf = io.BytesIO()\n'
    '        with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:\n'
    '            for p in files:\n'
    '                z.write(p, arcname=f"hasil_model/{p.name}")\n'
    '        st.download_button("Unduh hasil_model.zip", buf.getvalue(), "hasil_model.zip", "application/zip", icon=":material/folder_zip:")\n'
    '    st.caption("Hasil bersifat deskriptif dari data historis, bukan prediksi.")'
)
print(f"Kode dashboard siap: {len(dashboard_code.splitlines())} baris")

Kode dashboard siap: 801 baris


## 2 — Verifikasi dan buat file .py

Sel ini memeriksa struktur, sintaks, dan bahwa komponen yang diminta memang sudah hilang. File akan disimpan sebagai `dashboard_model.py`.

In [2]:
from pathlib import Path
import ast
import re

ast.parse(dashboard_code, filename="dashboard_model.py")
assert 'section("Insight utama"' not in dashboard_code
assert 'adjusted_rand_score' not in dashboard_code
assert not re.search(r"\bARI\b", dashboard_code, re.IGNORECASE)
for page in ["overview", "exploration", "distance", "clustering", "recommendation", "methodology"]:
    assert f"def page_{page}() -> None:" in dashboard_code

file_out = Path("dashboard_model.py")
if file_out.exists() and file_out.read_text(encoding="utf-8") != dashboard_code:
    backup = Path("dashboard_model_backup.py")
    backup.write_text(file_out.read_text(encoding="utf-8"), encoding="utf-8")
    print("Cadangan file lama disimpan di:", backup)
file_out.write_text(dashboard_code, encoding="utf-8")
print("BERHASIL:", file_out.resolve(), "—", file_out.stat().st_size, "bytes")
print("Insight utama dan tampilan ARI sudah dihapus; 6 halaman tetap ada.")

BERHASIL: /mnt/data/dashboard_model.py — 53418 bytes
Insight utama dan tampilan ARI sudah dihapus; 6 halaman tetap ada.


## 3 — Unduh dan pasang di GitHub

Jalankan sel ini untuk mengunduh `dashboard_model.py`. Di GitHub buka folder **`scripts`**, lalu **ganti `dashboard_model.py` lama** dengan berkas yang terunduh dan klik **Commit changes**. Refresh aplikasi Streamlit sesudah commit.

In [3]:
try:
    from google.colab import files
    files.download(str(file_out))
except ImportError:
    print("Jika tidak memakai Google Colab, file sudah tersedia di:", file_out.resolve())

Jika tidak memakai Google Colab, file sudah tersedia di: /mnt/data/dashboard_model.py
